In [46]:
import pandas as pd
from sklearn.decomposition import PCA
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import k_means
from fcmeans import FCM

import numpy as np
import plotly.io as pio
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist


import matplotlib.pyplot as plt

pio.renderers.default = 'vscode'

In [3]:
data = pd.read_csv("data/Measurements.csv", sep=";")
data.head(5)

,пол,"длина тела, см",длина туловища,длина ноги,"размах рук, см","масса тела, кг",окруж. груд.кл.в покое.,ширина плеч,ширина таза,обхват ягодиц
0,ж,161,48,84,160,"57,3","82,5","30,5",26,"97,5"
1,ж,"151,5",47,NaN,155,"52,4",77,37,28,85
2,м,182,53,NaN,196,71,88,42,33,90
3,м,181,54,NaN,191,84,94,47,32,96
4,м,169,51,NaN,180,"62,8",83,47,47,89


In [4]:
alias = {"М": "м", "муж": "м", "м":"м", "Ж": "ж", "жен": "ж", "ж":"ж"}

def gender_format(value: str):
    return alias[value]

data["пол"] = data["пол"].map(gender_format)
gender = data["пол"]
data["пол"].value_counts()


пол
ж    206
м     85
Name: count, dtype: int64

In [5]:
required_columns = ["длина тела, см", "масса тела, кг", "окруж. груд.кл.в покое.", "ширина плеч", "ширина таза"]

for col in required_columns:
    data[col] = data[col].astype(str).str.replace(",", ".", regex=False)

data = data.astype({col: float for col in required_columns})
data = data.dropna()
gender = data['пол']
data = data[required_columns]

In [13]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

pca = PCA(n_components=3)
data_pca = pca.fit_transform(data_scaled)

print(f"Размерность {data_pca.shape}")

(250, 3)


In [17]:
centroids, clusters_idx, inertion = k_means(data_pca, n_clusters=3)

In [21]:
df_plot_pca = pd.DataFrame({
    'PC1': data_pca[:, 0],
    'PC2': data_pca[:, 1],
    'PC3': data_pca[:, 2],
    'Кластер': clusters_idx.astype(str)
})

fig = px.scatter_3d(
    df_plot_pca,
    x='PC1',
    y='PC2',
    z='PC3',
    color='Кластер',
    title=f'PCA + Иерархическая кластеризация (K=3)',
    labels={'PC1': 'Главная компонента 1',
            'PC2': 'Главная компонента 2',
            'PC3': 'Главная компонента 3'},
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig.update_traces(marker=dict(size=5))
fig.update_layout(width=800, height=600)
fig.show()

In [43]:
fcm = FCM(n_clusters=3)
fcm.fit(data_pca)
membership_matrix = fcm.u 
centers = fcm.centers

In [42]:
membership_matrix

array([[0.21939573, 0.0345735 , 0.74603077],
       [0.51059956, 0.05178986, 0.43761059],
       [0.14554634, 0.01848533, 0.83596834],
       [0.05885054, 0.01003178, 0.93111767],
       [0.70180466, 0.0583699 , 0.23982544],
       [0.50116591, 0.04025402, 0.45858006],
       [0.47442928, 0.41933295, 0.10623778],
       [0.12839248, 0.82309633, 0.04851118],
       [0.59914275, 0.22386865, 0.1769886 ],
       [0.0793376 , 0.02081862, 0.89984378],
       [0.10809131, 0.02868141, 0.86322728],
       [0.13275995, 0.8172072 , 0.05003284],
       [0.95124665, 0.02322333, 0.02553002],
       [0.17683978, 0.01973814, 0.80342208],
       [0.70464983, 0.1319243 , 0.16342587],
       [0.89888685, 0.01912897, 0.08198418],
       [0.63212633, 0.27423466, 0.09363901],
       [0.28555438, 0.58980524, 0.12464038],
       [0.48092862, 0.04101471, 0.47805667],
       [0.10895582, 0.83998913, 0.05105505],
       [0.58512506, 0.31605708, 0.09881786],
       [0.38238282, 0.51259875, 0.10501843],
       [0.

In [45]:
hard_clusters = np.argmax(membership_matrix, axis=1)

df_plot_fcm = pd.DataFrame({
    'PC1': data_pca[:, 0],
    'PC2': data_pca[:, 1],
    'PC3': data_pca[:, 2],
    'Кластер': hard_clusters.astype(str),
    'Уверенность': np.max(membership_matrix, axis=1),  
    'К кластеру 0': membership_matrix[:, 0],
    'К кластеру 1': membership_matrix[:, 1],
    'К кластеру 2': membership_matrix[:, 2]
})


fig = px.scatter_3d(
    df_plot_fcm,
    x='PC1', y='PC2', z='PC3',
    color='Кластер',
    size='Уверенность',  # размер показывает степень принадлежности
    title='Fuzzy C-means кластеризация (размер = уверенность)',
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Set1,
    hover_data={'К кластеру 0': ':.2f', 
                'К кластеру 1': ':.2f', 
                'К кластеру 2': ':.2f'}
)
fig.update_layout(width=800, height=600)
fig.show()